# 03 · `gl_engine/domain/cell.py`

## What this file is for

The unit every number in this engine travels in.

A `Cell` is a value plus **where it came from**. The point is structural: `erc_source` is mandatory and has no default, so **a value that cannot name its source in ISO's files cannot be constructed at all**. The project's evidence rule isn't enforced by code review — it's enforced by the object refusing to exist.

**Depends on:** [`02-errors`](02-errors.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.domain import cell

for name, obj in vars(cell).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != cell.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Three dispositions, and they are not interchangeable.

In [ ]:
from gl_engine.domain.cell import Cell, Citation, Disposition

for d in Disposition:
    print(f"{d.name:<12} {d.value}")

- **`PUBLISHED`** — ISO filed a value, and here it is.
- **`NOT_OFFERED`** — ISO filed *nothing*, deliberately. An empty table means the coverage isn't offered here; it does **not** mean look somewhere else.
- **`REFER`** — ISO's content says a person decides.

Collapsing `NOT_OFFERED` into "zero" is the most expensive mistake available in this domain, because zero is a perfectly plausible premium.

## The interesting case

### You cannot build a value without a source

In [ ]:
cite = Citation(artifact="DedFactorProdsCSL.RateTable.csv",
                category="Rate Tables",
                locator="row 12",
                package="GL_CW_20260101_V01")

good = Cell.published(0.015, cite)
print("usable :", good.is_usable)
print("value  :", good.require_value())
print("source :", good.erc_source)

try:
    Cell(value=0.015)
except TypeError as e:
    print("\nwithout a source:", e)

### `require_value()` is where a refusal becomes an exception

A `REFER` cell is a legitimate object. It only raises when something tries to *use* it as a number — so the refusal travels intact right up to the moment it would have become a wrong premium.

In [ ]:
from gl_engine.errors import ReferToCompany

refer = Cell.refer(cite, "a per-claim deductible needs a company factor")
print("disposition:", refer.disposition.name)
print("is_usable  :", refer.is_usable)

try:
    refer.require_value()
except ReferToCompany as e:
    print("\nusing it raises:\n ", e)

## What it refuses

Two refusals, different in kind: one is *ours* (a value with no provenance is a programming error), one is *ISO's* (a value ISO says a person must supply).

In [ ]:
not_offered = Cell.not_offered(cite, "table is filed empty in this state")
print("NOT_OFFERED is_usable:", not_offered.is_usable)
try:
    not_offered.require_value()
except Exception as e:
    print(f"  {type(e).__name__}: {e}")

## Try it yourself

1. Where in the engine is `Cell` actually constructed? Everywhere, or only at particular boundaries?
2. What does `confirmed_by` hold, and which tier of the evidence hierarchy does it represent?
3. Build a `NOT_OFFERED` cell and a `PUBLISHED` cell holding `0`. What in the code distinguishes them downstream?

In [ ]:
# your turn